In [ ]:
# --- 1. INSTALL DEPENDENCIES FOR BOTH PIPELINES ---

# For your OCR Pipeline (Pipeline A)
print("Installing dependencies for OCR Pipeline...")
!sudo apt-get update && sudo apt-get install -y tesseract-ocr poppler-utils
!pip install -q pytesseract pdf2image opencv-python-headless rank_bm25

# For the ColQwen Pipeline (Pipeline B)
print("\nInstalling dependencies for ColQwen Pipeline...")
!pip install -q transformers accelerate bitsandbytes sentencepiece

print("\n✅ All dependencies installed.")```

---

### Cell 2: Build and Index Pipeline A (Your OCR Script + BM25 Search)

Here, we use your `slide_processor.py` to extract the text and then add the missing retrieval part using a classic, powerful search algorithm called `BM25`.

```python
# --- 2. BUILD AND INDEX PIPELINE A (OCR + BM25) ---

import os
from slide_processor import SlideOCR
from rank_bm25 import BM25Okapi
import time

print("--- Building Pipeline A: Your OCR Script + BM25 Search ---")

# --- Part 1: Run your script to extract text ---
# This uses your file exactly as provided.
SLIDE_DIR = "slides"
OCR_OUTPUT_DIR = "results/ocr"

print(f"Running OCR extraction on all PDFs in '{SLIDE_DIR}'...")
ocr_processor = SlideOCR(output_dir=OCR_OUTPUT_DIR)
ocr_results = ocr_processor.process_slide_directory(SLIDE_DIR)
print(f"\nOCR extraction complete. Processed {len(ocr_results)} pages.")

# --- Part 2: Build the Search Index (the missing retrieval part) ---
# Your script extracts text; BM25 makes it searchable.
corpus = []
doc_metadata = []
for result in ocr_results:
    corpus.append(result['text'])
    doc_metadata.append({'source': os.path.basename(result['source']), 'page': result['page']})

if not corpus:
    print("Error: No text was extracted. Cannot build search index for Pipeline A.")
    bm25 = None
else:
    tokenized_corpus = [doc.split(" ") for doc in corpus]
    print("\nBuilding BM25 search index...")
    start_time = time.time()
    bm25 = BM25Okapi(tokenized_corpus)
    end_time = time.time()
    print(f"✅ BM25 index built in {end_time - start_time:.2f} seconds.")

# --- Part 3: Create a search function for this pipeline ---
def search_ocr_pipeline(query, top_k=3):
    if bm25 is None:
        print("BM25 index not available.")
        return []
    tokenized_query = query.split(" ")
    doc_scores = bm25.get_scores(tokenized_query)
    
    # Get top k results where score is > 0
    top_n_indices = [i for i in doc_scores.argsort()[-top_k:][::-1] if doc_scores[i] > 0]
    
    results = []
    for i in top_n_indices:
        results.append({
            'score': doc_scores[i],
            'source': doc_metadata[i]['source'],
            'page': doc_metadata[i]['page'],
            'text_snippet': corpus[i][:250] + "..."
        })
    return results

In [ ]:
# --- 3. BUILD AND INDEX PIPELINE B (ColQwen) ---

import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForRetrieval
from pdf2image import convert_from_path
from pathlib import Path

print("\n--- Building Pipeline B: ColQwen Visual Retrieval ---")

# --- Part 1: Load the ColQwen model ---
print("Loading ColQwen model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained('vidore/colqwen2.5-v0.2', trust_remote_code=True)
model = AutoModelForRetrieval.from_pretrained(
    'vidore/colqwen2.5-v0.2', trust_remote_code=True
).eval().to(device)
print("✅ ColQwen model loaded.")

# --- Part 2: Build the Search Index ---
# For ColQwen, "indexing" means creating patch embeddings for each slide image.
colqwen_index = []
colqwen_images = {} # To store images for display

print(f"\nIndexing all PDFs in '{SLIDE_DIR}' with ColQwen...")
start_time = time.time()
slide_files = list(Path(SLIDE_DIR).glob('*.pdf'))

for pdf_path in slide_files:
    print(f"  - Processing {pdf_path.name}...")
    images = convert_from_path(str(pdf_path))
    for i, page_image in enumerate(images):
        doc_outputs = model.encode_document(page_image, processor)
        colqwen_index.append({
            'source': pdf_path.name,
            'page': i + 1,
            'embeddings': doc_outputs['doc_embeds']
        })
        colqwen_images[f"{pdf_path.name}_{i+1}"] = page_image

end_time = time.time()
print(f"✅ ColQwen index built for {len(colqwen_index)} pages in {end_time - start_time:.2f} seconds.")

# --- Part 3: Create a search function for this pipeline ---
def search_colqwen_pipeline(query, top_k=3):
    query_embeds = model.encode_query(query, processor)['query_embeds'].to(device)
    
    page_scores = []
    for doc in colqwen_index:
        doc_embeds = doc['embeddings'].to(device)
        score = model.score(query_embeds, doc_embeds).item()
        page_scores.append(score)
    
    top_n_indices = sorted(range(len(page_scores)), key=lambda i: page_scores[i], reverse=True)[:top_k]
    
    results = []
    for i in top_n_indices:
        doc_info = colqwen_index[i]
        results.append({
            'score': page_scores[i],
            'source': doc_info['source'],
            'page': doc_info['page'],
            'image': colqwen_images[f"{doc_info['source']}_{doc_info['page']}"]
        })
    return results

In [ ]:
# --- 4. HEAD-TO-HEAD EVALUATION ---
from IPython.display import display

# Define a set of diverse queries to test both systems
queries = [
    "tesseract setup",                            # Specific keywords
    "slide showing a diagram of the model",       # Visual / Conceptual query
    "table with performance metrics",             # Layout / Visual query
    "cv2.COLOR_BGR2GRAY",                         # Code / Exact string query
    "summary of the methodology",                 # Semantic query
]

for query in queries:
    print("="*80)
    print(f"PERFORMING QUERY: '{query}'")
    print("="*80)
    
    # --- Pipeline A Results ---
    print("\n--- ✅ Results from Pipeline A (Your OCR + BM25) ---")
    ocr_search_results = search_ocr_pipeline(query)
    if not ocr_search_results:
        print("No results found.")
    for res in ocr_search_results:
        print(f"  Source: {res['source']}, Page: {res['page']} (Score: {res['score']:.2f})")
        print(f"  Snippet: {res['text_snippet']}\n")

    # --- Pipeline B Results ---
    print("\n--- ✅ Results from Pipeline B (ColQwen) ---")
    colqwen_search_results = search_colqwen_pipeline(query)
    if not colqwen_search_results:
        print("No results found.")
    for res in colqwen_search_results:
        print(f"  Source: {res['source']}, Page: {res['page']} (Score: {res['score']:.4f})")
        # Display the actual image result from ColQwen
        display(res['image'].resize((400, 300)))
    
    print("\n\n")